# Document Classifier — Training & Artifact Summary

This notebook documents the deep-learning classifier shipped in `backend/app/classifier/` and re-verifies its artifacts.

**It does not train** and **does not download** the full RVL-CDIP dataset. Safe to run on any dev machine with the virtualenv active from `backend/`.

## Dataset Summary

**RVL-CDIP** — Ryerson Vision Lab Complex Document Image Processing dataset.

- 400,000 grayscale TIFF images across 16 document-layout classes
- Official split: 320k train / 40k validation / 40k test
- Task: visual/layout classification — **no OCR**, no text extraction

Classes (in label-ID order as committed in `model_card.json`):

| ID | Class |
|----|-------|
| 0 | letter |
| 1 | form |
| 2 | email |
| 3 | handwritten |
| 4 | advertisement |
| 5 | scientific_report |
| 6 | scientific_publication |
| 7 | specification |
| 8 | file_folder |
| 9 | news_article |
| 10 | budget |
| 11 | invoice |
| 12 | presentation |
| 13 | questionnaire |
| 14 | resume |
| 15 | memo |

Source TIFFs are grayscale; inference must call `image.convert("RGB")` before the preprocessing pipeline because ConvNeXt expects a 3-channel input.

## Why Colab Was Used

The full RVL-CDIP dataset (~40 GB) is too large for local download during development.

Training used:
- **Google Colab GPU** for compute
- **HuggingFace `datasets`** with streaming mode — individual images are fetched on demand; the full dataset is never materialised locally

The repo ships only the trained artifact, model card, and verification logic. No training loop, no dataset download is required at runtime.

## Model Architecture

| Component | Detail |
|-----------|--------|
| Backbone | ConvNeXt-Tiny |
| Pretrained weights | `ConvNeXt_Tiny_Weights.DEFAULT` (ImageNet-1K) |
| Classifier head | `nn.Linear(768, 16)` replacing the original 1000-class head |
| Freeze policy | Partial unfreeze: `features[-1]` (final ConvNeXt stage) + classifier head |
| Trainable params | ~14.3M of ~27.8M total (~51%) |

The final ConvNeXt stage (`features[-1]`) is the largest single block (~12M params), which is why unfreezing it alone accounts for roughly half the trainable parameters. Earlier stages stay frozen to preserve general low-level ImageNet features.

## Static Training Configuration

| Hyperparameter | Value |
|----------------|-------|
| Backbone | convnext_tiny |
| Pretrained weights | ConvNeXt_Tiny_Weights.DEFAULT |
| Freeze policy | partial_unfreeze (features[-1] + classifier) |
| Image size | 224×224 (final tensor to ConvNeXt) |
| Epochs | 2 |
| Batch size | 32 |
| Train steps per epoch | 10,000 (streaming, ~320k images/epoch) |
| Optimizer | AdamW (weight_decay=1e-4) |
| Learning rate | 3e-4 (epoch 1), 1e-4 (epoch 2) |
| Train augmentation | Resize(256) → RandomResizedCrop(224, scale=0.85–1.0) → RandomRotation(2°) |
| Eval / inference preprocessing | Resize(236, shorter side) → CenterCrop(224) → ToTensor → Normalize(ImageNet) |

## Training History

Values from `model_card.json training_history_summary.records`:

| Epoch | Train Loss | Train Acc | Val Loss | Val Top-1 | Val Top-5 |
|-------|-----------|-----------|----------|-----------|----------|
| 1 | 0.7150 | 0.7831 | 0.7285 | 0.7833 | 0.9591 |
| 2 | 0.4418 | 0.8659 | 0.6751 | 0.8018 | 0.9628 |

Epoch 2 was selected as the final checkpoint (best `val_top1 = 0.8018`).

In [ ]:
import json
from pathlib import Path

import pandas as pd

card = json.loads(Path("../app/classifier/models/model_card.json").read_text(encoding="utf-8"))

ft = card["metrics"]["full_test"]
print("=== Full-test evaluation (epoch 2 checkpoint) ===")
print(f"  SHA-256 : {card['checkpoint']['sha256']}")
print(f"  Device  : {ft['device']}")
print(f"  Loss    : {ft['loss']:.6f}")
print(f"  Top-1   : {ft['top1']:.6f}")
print(f"  Top-5   : {ft['top5']:.6f}")
print(f"  Examples evaluated : {ft['num_examples_evaluated']} / {ft['num_examples_expected']}")
print(f"  Skipped            : {ft['num_examples_skipped']}")
print()

df = pd.DataFrame.from_dict(ft["per_class_accuracy"], orient="index", columns=["accuracy"])
df.index.name = "class"
df = df.sort_values("accuracy")
df["accuracy_fmt"] = df["accuracy"].map("{:.4f}".format)

print("Per-class accuracy (sorted ascending):")
styled = (
    df[["accuracy_fmt"]]
    .rename(columns={"accuracy_fmt": "accuracy"})
    .style.highlight_min(subset=["accuracy"], color="#ffcccc")
)
display(styled)

In [ ]:
import json
from collections import Counter
from pathlib import Path

entries = json.loads(
    Path("../app/classifier/eval/golden_expected.json").read_text(encoding="utf-8")
)

classes_covered = sorted({e["true_label"] for e in entries})
reason_counts = Counter(e["selection_reason"] for e in entries)

print(f"Total golden entries : {len(entries)}")
print(f"Classes covered      : {len(classes_covered)}")
print()
print("Selection reason breakdown:")
for reason, count in sorted(reason_counts.items()):
    print(f"  {reason:<55s} {count}")

print()
print("5 hardest goldens by ascending top1-top2 margin:")
hardest = sorted(entries, key=lambda e: e["top1_top2_margin"])[:5]
for e in hardest:
    print(
        f"  {e['filename']:<45s}  margin={e['top1_top2_margin']:.4f}  "
        f"label={e['true_label']}  conf={e['expected_top1_confidence']:.4f}"
    )

### What the frozen golden-set outputs protect

The CI golden-set replay (`python -m app.classifier.eval.golden`) detects:

| Drift type | How it shows up |
|------------|-----------------|
| Preprocessing change | Confidence values shift; hard examples may flip label |
| Model file swap / corruption | Labels and confidences change entirely |
| Class-order change | Labels report correct index but wrong name |
| Head re-initialisation | Weights differ; all confidences scrambled |
| Accidental backbone swap | Similar — large-scale confidence shifts |

All 50 expected outputs (label, top-1 confidence ±1e-6, full top-5 order) must match exactly on every CI run. Any drift blocks the merge.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from app.classifier.model import verify_artifacts

try:
    card = verify_artifacts()
    print("PASS: artifacts verified")
    print(f"  SHA-256   : {card['checkpoint']['sha256']}")
    print(f"  Test top-1: {card['metrics']['full_test']['top1']:.4f}")
except Exception as e:
    print(f"FAIL: {e}")
    raise

## Golden-Set Replay

Command (run from `backend/`):

```bash
uv run python -m app.classifier.eval.golden
```

What it asserts for each of the 50 golden images:
- **Label exact match** — `pred.label_name == entry["expected_label"]`
- **Top-1 confidence within 1e-6** — `abs(pred.top1_confidence - entry["expected_top1_confidence"]) <= 1e-6`
- **Full top-5 ordering match** — the ordered list of 5 label names must be identical

All three conditions must hold for all 50 images. Any mismatch prints `FAIL` and exits 1, blocking CI. Expected values in `golden_expected.json` were generated using the production preprocessing pipeline (`Resize(236) → CenterCrop(224)`) with `CLASSIFIER_TORCH_NUM_THREADS=1` to match the CI environment.

In [ ]:
# Optional — already covered by CI. Run manually to confirm local environment.
# Requires the virtualenv to be active and the working directory to be backend/.

import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "app.classifier.eval.golden"],
    capture_output=True,
    text=True,
)
print(result.stdout or result.stderr)
if result.returncode != 0:
    raise RuntimeError("Golden replay failed — see output above.")

## Inference Latency Benchmark

Command (run from `backend/`):

```bash
uv run python -m app.classifier.benchmark_inference_latency
```

What it measures:
- Runs inference on all 50 golden images with configurable warmup and repeats
- Records per-image latency; reports mean, p50, **p95**, and max
- **Budget: p95 < 1000 ms** on CPU
- Exits 1 if p95 ≥ budget; writes `eval/benchmark_report.json` and `eval/benchmark_report.txt`

The benchmark uses the same golden images and the same `predict_image_path()` pipeline as the golden-set replay, so it reflects real production preprocessing latency.

In [ ]:
import json
from pathlib import Path

report_path = Path("../app/classifier/eval/benchmark_report.json")

if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("=== Classifier Inference Latency Report ===")
    print(f"  Device          : {report['device']}")
    print(f"  Torch threads   : {report['torch_num_threads']}")
    print(f"  Torch version   : {report['torch_version']}")
    print(f"  Images tested   : {report['num_images']} x {report['repeats_per_image']} repeat(s)")
    print(f"  Warmup runs     : {report['warmup_runs_not_counted']} (not counted)")
    print()
    print(f"  mean  {report['mean_ms']:.2f} ms")
    print(f"  p50   {report['p50_ms']:.2f} ms")
    print(f"  p95   {report['p95_ms']:.2f} ms")
    print(f"  max   {report['max_ms']:.2f} ms")
    print()
    verdict = "PASS" if report["pass"] else "FAIL"
    print(f"  Budget : p95 < {report['budget_ms']:.0f} ms")
    print(f"  Result : {verdict}  ({report['p95_ms']:.2f} ms)")
else:
    print("No benchmark_report.json found. Run the benchmark first:")
    print("  uv run python -m app.classifier.benchmark_inference_latency")

In [ ]:
from pathlib import Path

classifier_dir = Path("../app/classifier")

print("Shipped classifier artifacts:")
for p in sorted(classifier_dir.rglob("*")):
    if p.is_file() and "__pycache__" not in p.parts and not p.name.startswith("."):
        rel = p.relative_to(classifier_dir)
        size_kb = p.stat().st_size / 1024
        print(f"  {str(rel):<65s}  {size_kb:>8.1f} KB")

## Defense Notes

1. **Local stack does not train.** The repo ships the trained checkpoint. No training loop is present in the backend package.

2. **Local stack does not download the full RVL-CDIP dataset.** Training used HuggingFace streaming on Colab. Runtime only needs the 50 golden TIFFs in `eval/golden_images/`.

3. **API and worker refuse to boot on bad artifacts.** `verify_artifacts()` is called at startup and raises on: missing weights, SHA-256 mismatch, or `metrics.full_test.top1 < 0.70`.

4. **Golden-set replay runs in CI and blocks merges on any drift.** The 50 frozen expected outputs catch preprocessing drift, model file corruption, class-order changes, and accidental head or backbone swaps.

5. **Full-test top-1 = 0.803**, above the 0.70 quality gate. Weakest class: `scientific_report` at **0.516** — most commonly confused with `presentation`.